In [0]:
dbutils.widgets.text("catalog", "shopstream_dev")
catalog = dbutils.widgets.get("catalog")

In [0]:
%sql
SELECT status, COUNT(*) FROM shopstream_dev.bronze.orders GROUP BY status
ORDER BY 2 DESC;

In [0]:
%sql

SELECT
(SELECT COUNT(*) FROM (SELECT order_line_id FROM
shopstream_dev.bronze.orders
GROUP BY order_line_id HAVING COUNT(*) > 1)) AS
duplicated_line_ids,
(SELECT COUNT(*) FROM shopstream_dev.bronze.orders WHERE quantity <= 0) AS
bad_quantity_rows;

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.silver.orders AS
WITH ranked AS (
SELECT *, ROW_NUMBER() OVER (PARTITION BY order_line_id ORDER BY order_ts)
AS rn
FROM {catalog}.bronze.orders
)
SELECT order_line_id, order_id, customer_id, product_id,
CAST(quantity AS INT) AS quantity,
CAST(unit_price AS DOUBLE) AS unit_price,
ROUND(quantity * unit_price, 2) AS line_revenue,
CAST(order_ts AS TIMESTAMP) AS order_ts,
LOWER(status) AS status,
NULLIF(coupon_code, '') AS coupon_code
FROM ranked
WHERE rn = 1 AND quantity > 0
""")

In [0]:
spark.sql(f"DELETE FROM {catalog}.silver.orders WHERE status = 'cancelled'")
display(spark.sql(f"""
SELECT (SELECT COUNT(*) FROM {catalog}.silver.orders) AS rows_now,
(SELECT COUNT(*) FROM {catalog}.silver.orders VERSION AS OF 0) AS
rows_before_delete
"""))

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.silver.products AS
SELECT product_id, product_name, category,
CAST(unit_price AS DOUBLE) AS unit_price,
CAST(unit_cost AS DOUBLE) AS unit_cost,
ROUND(unit_price - unit_cost, 2) AS unit_margin
FROM {catalog}.bronze.products
""")

In [0]:
%sql
DESCRIBE HISTORY shopstream_dev.silver.orders

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.silver.customers (
customer_id STRING, name STRING, email STRING, phone STRING,
street_address STRING, postcode STRING, city STRING, country STRING,
date_of_birth DATE, signup_date DATE, signup_channel STRING,
effective_from TIMESTAMP, effective_to TIMESTAMP, is_current BOOLEAN)
""")
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW customers_latest AS
SELECT customer_id, name, email, CAST(phone AS STRING) AS phone,
street_address,
CAST(postcode AS STRING) AS postcode, city, country,
CAST(date_of_birth AS DATE) AS date_of_birth,
CAST(signup_date AS DATE) AS signup_date, signup_channel
FROM (
SELECT *, ROW_NUMBER() OVER (PARTITION BY customer_id
ORDER BY _ingested_at DESC, _source_file DESC)
AS rn
FROM {catalog}.bronze.customers
) WHERE rn = 1
""")

In [0]:
spark.sql(f"""
MERGE INTO {catalog}.silver.customers t
USING customers_latest s
ON t.customer_id = s.customer_id AND t.is_current = true
WHEN MATCHED AND (t.street_address <> s.street_address OR t.postcode <>
s.postcode
OR t.email <> s.email OR t.phone <> s.phone OR t.city <>
s.city) THEN
UPDATE SET is_current = false, effective_to = current_timestamp()
WHEN NOT MATCHED THEN
INSERT (customer_id, name, email, phone, street_address, postcode, city,
country,
date_of_birth, signup_date, signup_channel, effective_from,
effective_to, is_current)
VALUES (s.customer_id, s.name, s.email, s.phone, s.street_address,
s.postcode, s.city,
s.country, s.date_of_birth, s.signup_date, s.signup_channel,
current_timestamp(), NULL, true)
""")
spark.sql(f"""
INSERT INTO {catalog}.silver.customers
SELECT s.*, current_timestamp(), NULL, true
FROM customers_latest s
WHERE NOT EXISTS (SELECT 1 FROM {catalog}.silver.customers t
WHERE t.customer_id = s.customer_id AND t.is_current =
true)
""")

In [0]:
%sql
SELECT * FROM shopstream_dev.silver.customers

In [0]:
%sql
SELECT COUNT(*) AS all_rows, SUM(CAST(is_current AS INT)) AS current_rows
FROM shopstream_dev.silver.customers; -- expect 1,020 and 1,000


In [0]:
%sql
SELECT customer_id, street_address, effective_from, effective_to, is_current
FROM shopstream_dev.silver.customers
WHERE customer_id IN (SELECT customer_id FROM shopstream_dev.silver.customers
WHERE NOT is_current)
ORDER BY customer_id, effective_from;


In [0]:
spark.sql(f"""
CREATE FUNCTION IF NOT EXISTS {catalog}.silver.mask_text(val STRING)
RETURNS STRING
RETURN CASE
    WHEN is_account_group_member('pii_readers')
    THEN val
    ELSE '***MASKED***'
END
""")

In [0]:
spark.sql(f"""
CREATE FUNCTION IF NOT EXISTS {catalog}.silver.mask_date(val DATE)
RETURNS DATE
RETURN CASE
    WHEN is_account_group_member('pii_readers')
    THEN val
    ELSE NULL
END
""")

In [0]:
for col in ["email", "phone", "street_address"]:
    spark.sql(f"""
        ALTER TABLE {catalog}.silver.customers
        ALTER COLUMN {col}
        SET MASK {catalog}.silver.mask_text
    """)

In [0]:
spark.sql(f"""
    ALTER TABLE {catalog}.silver.customers
    ALTER COLUMN date_of_birth
    SET MASK {catalog}.silver.mask_date
""")


In [0]:
%sql
SELECT
    customer_id,
    email,
    phone,
    street_address,
    date_of_birth
FROM shopstream_dev.silver.customers
LIMIT 5;